DL2A-lab-conv2D-2025.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1o61ieyhjoDGkbKFy8hEdz5X8JXjc9ct6

This notebook is  a follow up of the lab session on Fashion MNIST. Be sure you did and understood the previous notebook on applying the feed-forward neural network with pytorch.

The goal is to incrementally build an image classifier based on convolutional layers. Since we consider images and convolution we will use Tensors with peculiar shapes in input. Moreover, this session is also the occasion to introduce "Max-pooling" and to sue "Batch-normalization" again.




# Dataset

First get the dataset. This is the same as last lab.

In [ ]:
# math, numpy and plot
import numpy as np
import math
import time
import matplotlib
import matplotlib.pyplot as plt
# torch
import torch as th
import torch.autograd as autograd
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# gzip
import gzip
import pickle

In [ ]:
# To download the file:
import os
if not os.path.exists("fashion-mnist.pk.gz"):
    import gdown
    gdown.download("https://drive.google.com/uc?id=1qx-a3KjzX2W66Aq84tnjGRUa10CTxOJN",
                   "fashion-mnist.pk.gz", quiet=False)

# Or if you prefer from Colab:
#import gdown
#url = 'https://drive.google.com/uc?id=1qx-a3KjzX2W66Aq84tnjGRUa10CTxOJN'
#output = 'fashion-mnist.pk.gz'
#gdown.download(url, output, quiet=False)
# Load the dataset
fp = gzip.open('./fashion-mnist.pk.gz','rb')
# ATTENTION : le pickle contient (Xtrain, Ytrain, Xtest, Ytest, classlist).
# Les 3e et 4e elements sont bien le jeu de TEST, pas la validation : la
# validation, on va la decouper nous-memes dans les donnees d'entrainement.
allXtrain, allYtrain, Xtest, Ytest, classlist = pickle.load(fp)
print("train complet :", allXtrain.shape, "| test :", Xtest.shape)
print("classes :", classlist)

##  Data format

The provided version is unfortunately not adapted to our purpose in terms of dimensions.
*Convolution2D* expects as input a Tensor with 4 dimensions $(N,C,H,W)$ with :

- N the batch dimension, *i.e* the number of images
- C the number of input channels, here it is 1
- H the height or number of rows of each image
- W the width  or number of columns of each image

## DataSet and DataLoader

In pytorch data handling is done in 2 steps:
- `DataSet`: a class to access the raw data, it can be tensors, files, distributed files, ...
- `DataLoader` (the class to iterate through the dataset and to get access to well prepared batch of data)

From the model viewpoint:
- During training and testing the model interacts with the `DataLoader` to go through the `DataSet`
- The Dataloader pick what is necessary in the `DataSet`  

Here we have Tensors, so we can use a `TensorDataSet`.  This abstraction for handling data is important. Depending on the application and the data types, it is very easier in practice to divide the process in two steps: the Dataset and the interaction between the model and the data via the dataset.

In [ ]:
BatchSize = 100
splittrain = 30000

# On met directement les images au format attendu par Conv2d : (N, C, H, W).
# C'est plus propre que de faire un unsqueeze(1) dans la boucle d'entrainement.
Xtr = allXtrain[:splittrain].view(-1, 1, 28, 28) / 255
Xva = allXtrain[splittrain:].view(-1, 1, 28, 28) / 255

# create your datset
# dataset = TensorDataset(x_tensor, y_tensor)
trainset = TensorDataset(Xtr, allYtrain[:splittrain])
# create your dataloader
trainloader = DataLoader(trainset, batch_size=BatchSize, shuffle=True)
# Do the same for the validation set
validset = TensorDataset(Xva, allYtrain[splittrain:])
# shuffle=False en validation : l'ordre n'a aucune importance pour evaluer,
# et ne pas melanger rend les resultats reproductibles d'un run a l'autre.
validloader = DataLoader(validset, batch_size=BatchSize, shuffle=False)

print("train :", Xtr.shape, "| validation :", Xva.shape)

To look at one batch :

In [ ]:
batch = next(iter(trainloader))
# Explore what you get as a batch
print(batch[0].shape)
print(batch[1].shape)

# Playing with convolution in 2D

Let start the exploration of convolution.The class we will use is called Conv2d. Read carefully the documentation of this module. Maybe you cannot understand everything. That's why it is useful to first play with the convolution with one image.

In [ ]:
# Extract one image to start and be sure you have the right dimensions.
image=batch[0][0]
print(image.shape)
plt.imshow(image.squeeze(),  matplotlib.pyplot.cm.gray)

Now create one convolution layer with 1 input channels, a kernel size of 3, and a stride of 1.
Try it and look at the output dimension.

In [ ]:
# TODO:
convol=nn.Conv2d(in_channels=1,out_channels=1,kernel_size =3)
res=convol(image.unsqueeze(0).unsqueeze(0))
print(image.unsqueeze(0).shape)
print(image.unsqueeze(0).unsqueeze(0).shape)
print(res.shape)

The resulting "image" is not of the same dimension, how to obtain an output with the same dimension (same question with a kernel size of 5) ?

In [ ]:
# TODO
#need to put a padding
convol1=nn.Conv2d(in_channels=1,out_channels=1,kernel_size =3,padding=1)
res1=convol1(image.unsqueeze(0).unsqueeze(0))
print(res1.shape)

#froa kernel 5
convol2=nn.Conv2d(in_channels=1,out_channels=1,kernel_size =5,padding=2)
res2=convol2(image.unsqueeze(0).unsqueeze(0))
print(res2.shape)

We can define the parameters of the convolutional filter with our own hands. For that purpose we just have to create the Tensor we want and cast it in a *Parameter* object (usefull for autograd) and then assign it.
This is an example:

In [ ]:
# Create a convolutional filter
convFilter = nn.Conv2d(in_channels=1,out_channels=1,
                       kernel_size = 3, padding=1,
                       stride=1)
# build the weight matrix you want
W=th.ones(convFilter.weight.shape)
# Makes it a Parameter and assign
convFilter.weight = nn.Parameter(W)
im=image.unsqueeze(0).unsqueeze(0)

res = convFilter(im)
print(res.shape)
plt.subplot(1,2,1)
plt.imshow(im.squeeze(),  matplotlib.pyplot.cm.gray)
plt.subplot(1,2,2)
plt.imshow(F.relu(res).squeeze().detach(), matplotlib.pyplot.cm.gray)

print(W)

Now try to set the convolution fiter as follows:
$$
\left(
\begin{array}{ccc}
 -1 &2&-1\\
 -1 &2&-1\\
 -1 &2&-1
\end{array}
\right)
$$
and then as follows:
$$
\left(
\begin{array}{rrr}
 -1 &-1&-1\\
 2 &2&2\\
 -1 &-1&-1
\end{array}
\right)
$$
Try them on some images and visualize the results.

In [ ]:
# TODO

In [ ]:
convFilter1 = nn.Conv2d(in_channels=1,out_channels=1,
                       kernel_size =3,padding=1)
W1=th.tensor([[-1,2,-1],[-1,2,-1],[-1,2,-1]],dtype=th.float32).unsqueeze(0).unsqueeze(0)
convFilter1.weight = nn.Parameter(W1)

convFilter2= nn.Conv2d(in_channels=1,out_channels=1,
                       kernel_size =3,padding=1)
W2=th.tensor([[-1,-1,-1],[2,2,2],[-1,-1,-1]],dtype=th.float32).unsqueeze(0).unsqueeze(0)
convFilter2.weight = nn.Parameter(W2)

In [ ]:
plt.subplot(1, 3, 1)
plt.imshow(im.squeeze(), cmap='gray')

In [ ]:
plt.subplot(1, 3, 2)
plt.imshow(convFilter1(im).squeeze().squeeze().detach().numpy(), cmap='gray')

In [ ]:
plt.subplot(1, 3, 3)
plt.imshow(convFilter2(im).squeeze().squeeze().detach().numpy(), cmap='gray')

In [ ]:
plt.show()

# Pool !

Now we introduce the max-pooling in 2 dimensions: *MaxPool2d*. Look at the documentation and then try to define the following pipeline:
- a convolution with a kernel size of 3, stride 1 and padding 1
- apply the ReLu function and
- a max pooling with kernel size of 2 and a stride of 2.
Try to guess before running your code the dimensions of the output !

In [ ]:
# TODO

convFilter = nn.Conv2d(in_channels=1,out_channels=1,
                       kernel_size =3,padding=1)
res=F.relu(convFilter(im))
pool=nn.MaxPool2d(kernel_size=2,stride=2)
res1=pool(res)
plt.subplot(1, 3, 1)
plt.imshow(im.squeeze(), cmap='gray')
plt.subplot(1, 3, 2)
plt.imshow(res.squeeze().detach().numpy(), cmap='gray')
plt.subplot(1, 3, 3)
plt.imshow(res1.squeeze().detach().numpy(), cmap='gray')

#  A first model

After this interlude, the goal now is to write a class to implement the model with:

- 2D convolution with (kernel size = 3, padding = 1, stride 1)
- ReLu activation
- Max-pooling (kernel size = 2, stride 2)
- A final linear classifier
- The final activation

Writing this class, allows you to wrap what you have seen so far. To debug the model, you can first play step-by-step with each layer to ensure you obtain the right dimensions (it was done earlier). Then, write the class and run the training to evaluate the result (this what we have to do now).

The class inherits from an existing class of pytorch : *Module*. This mean: it is a *Module*, but we add some peculiarities. For that purpose we can fill the following code:

In [ ]:
class FashionCNN(nn.Module):

    def __init__(self, kernel_size = 3, padding= 1, out_channels = 1):
        super(FashionCNN, self).__init__()
        # TODO : write the end of the constructor.
        # It is important to create here all the layers of the network.
        # All layers that have paramaters should be attribute.
        # For example:

        self.conv = nn.Conv2d(in_channels=1, out_channels=out_channels,
                              kernel_size=kernel_size, padding=padding)
        # TODO: add the rest
        self.relu=nn.ReLU()
        self.pool=nn.MaxPool2d(kernel_size=2,stride=2)
        self.flat=nn.Flatten(start_dim=-3, end_dim=-1)
        self.fc=nn.Linear(in_features=out_channels * 14 * 14,out_features=10)
        self.logsoftmax=nn.LogSoftmax(dim=1)

In [ ]:
    def forward(self, x):
        # TODO
        # if you need to run forward with the conv layer,
        # you can call it by self.conv
        outconv  = self.conv(x)
        outrelu=self.relu(outconv)
        outpool=self.pool(outrelu)
        outflat=self.flat(outpool)
        outfc=self.fc(outflat)
        output=self.logsoftmax(outfc)
        return output

# Test the class: is everything in place:
# A first classifier is built like :
classif = FashionCNN(out_channels=4)
# The parameters of the classifier are randomly initialize, but we
# can use it on a image :
out = classif(im)                 # equivalent a classif.forward(im), mais c'est
                                  # la forme a utiliser : __call__ declenche aussi
                                  # les hooks de PyTorch, pas forward() directement
print(out.shape)                  # attendu : (1, 10) -> 1 image, 10 log-probas
print(out)
print("somme des probas :", out.exp().sum().item(), "(doit valoir 1)")

# Et sur un vrai batch, pour verifier que les dimensions suivent :
batch_x, batch_y = next(iter(trainloader))
print("\nbatch d'entree :", batch_x.shape)
print("sortie du modele :", classif(batch_x).shape, "-> (B, 10)")

# Suivi des dimensions a l'interieur du reseau, etape par etape.
# C'est LE reflexe a avoir quand un modele convolutif ne compile pas :
with th.no_grad():
    t = batch_x
    print("\n--- suivi des dimensions ---")
    print(f"entree            : {tuple(t.shape)}")
    t = classif.conv(t);  print(f"apres conv (k=3,p=1): {tuple(t.shape)}  <- H,W inchanges grace au padding")
    t = classif.relu(t);  print(f"apres relu        : {tuple(t.shape)}")
    t = classif.pool(t);  print(f"apres maxpool 2x2 : {tuple(t.shape)}  <- H,W divises par 2")
    t = classif.flat(t);  print(f"apres flatten     : {tuple(t.shape)}  <- 4*14*14 = {4*14*14}")
    t = classif.fc(t);    print(f"apres linear      : {tuple(t.shape)}")

# Training the model

To train the model, we need to define a loss function and an optimizer. For the moment we will rely on an online learning algorithm: online stochastic gradient descent. Like the previous lab session:
- we pick one training example
- compute the loss
- back-propagation of the gradient
- update of the parameters


At the end of one epoch, we evaluate the model on the validation step. You can use for that purpose the training function we wrote earlier.


To train the CNN, we can reuse the training function you wrote carefully in the previous lab session. However we need to adapt it in order to use dataloaders (the `trainloader` and `validloader`)

Question:
- As optimizer we will use *Adam*. It is important to find the good choice of hyper-parameter for the initial learning rate. Try different values like 0.1, 0.01, ...
- Then try with a number of output channel set to 1, 8, 16.

In [ ]:
# TODO : paste here the training function you wrote before.
# On la reecrit en version generique : elle prend des DataLoaders (et non des
# tenseurs entiers), gere le device (CPU/GPU) et suit train + validation.

device = th.device("cuda" if th.cuda.is_available() else "cpu")
print("device :", device)

In [ ]:
@th.no_grad()
def evaluate(model, loader, loss_function):
    """Renvoie (loss moyenne, accuracy) sur un DataLoader complet."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        out = model(X)
        # on pondere par la taille du batch : le dernier batch peut etre plus
        # petit, une moyenne des moyennes serait legerement fausse
        total_loss += loss_function(out, Y).item() * len(X)
        correct += (out.argmax(dim=1) == Y).sum().item()
        n += len(X)
    return total_loss / n, correct / n

In [ ]:
def train_model(model, loss_function, optimizer, trainloader, NEpochs,
                validloader=None, plot=True, label="", verbose=True):
    """Entraine un modele convolutif et renvoie l'historique."""
    if validloader is None:
        validloader = globals()['validloader']
    model = model.to(device)
    hist = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
    t0 = time.time()

    for epoch in range(NEpochs):
        model.train()
        running, correct, n = 0.0, 0, 0
        for X, Y in trainloader:
            X, Y = X.to(device), Y.to(device)
            out = model(X)
            loss = loss_function(out, Y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            running += loss.item() * len(X)
            correct += (out.argmax(dim=1) == Y).sum().item()
            n += len(X)

        hist['train_loss'].append(running / n)
        hist['train_acc'].append(correct / n)
        vl, va = evaluate(model, validloader, loss_function)
        hist['valid_loss'].append(vl); hist['valid_acc'].append(va)

        if verbose:
            print(f"epoch {epoch:3d} | train loss {running/n:.4f} acc {correct/n:.4f}"
                  f" | valid loss {vl:.4f} acc {va:.4f}")

    hist['time'] = time.time() - t0
    if plot:
        fig, axs = plt.subplots(1, 2, figsize=(11, 4))
        axs[0].plot(hist['train_loss'], 'r', label='train')
        axs[0].plot(hist['valid_loss'], 'orange', ls='--', label='validation')
        axs[0].set_title('Loss'); axs[0].set_xlabel('epoch')
        axs[1].plot(hist['train_acc'], 'g', label='train')
        axs[1].plot(hist['valid_acc'], 'darkgreen', ls='--', label='validation')
        axs[1].set_title('Accuracy'); axs[1].set_xlabel('epoch')
        for a in axs: a.grid(True, ls='--', alpha=.6); a.legend()
        fig.suptitle(f"{label}   ({hist['time']:.1f} s)")
        plt.tight_layout(); plt.show()
    return hist

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# TODO
loss_fn = nn.NLLLoss()

### Choix du learning rate pour Adam

On teste plusieurs valeurs. Adam adapte le pas par parametre, il tolere donc une
plage plus large que SGD, mais 0.1 reste beaucoup trop grand.

In [ ]:
for lr in (0.1, 0.01, 0.001, 0.0001):
    th.manual_seed(0)
    model = FashionCNN(out_channels=4)
    optimizer = th.optim.Adam(model.parameters(), lr=lr)
    h = train_model(model, loss_fn, optimizer, trainloader, 3, plot=False, verbose=False)
    print(f"lr = {lr:<7} -> valid acc apres 3 epochs = {h['valid_acc'][-1]:.4f}")

`lr = 1e-3` est le meilleur compromis : c'est la valeur par defaut d'Adam, et
c'est presque toujours le bon point de depart. Avec `lr = 0.1` l'entrainement
diverge ou stagne : les pas sont si grands qu'on saute par-dessus les minima.

### Influence du nombre de canaux de sortie

Chaque canal de sortie est un **filtre** different, donc un detecteur de motif
different (bord vertical, bord horizontal, coin, texture...). Plus on en a, plus
le reseau peut reperer de motifs differents.

In [ ]:
for oc in (1, 8, 16):
    th.manual_seed(0)
    model = FashionCNN(out_channels=oc)
    optimizer = th.optim.Adam(model.parameters(), lr=0.001)
    h = train_model(model, loss_fn, optimizer, trainloader, 5, plot=False, verbose=False)
    print(f"out_channels = {oc:2d} | {count_params(model):7d} parametres"
          f" | valid acc = {h['valid_acc'][-1]:.4f}")

th.manual_seed(0)
model = FashionCNN(out_channels=8)
optimizer = th.optim.Adam(model.parameters(), lr=0.001)
hist_base = train_model(model, loss_fn, optimizer, trainloader, 10,
                        label="FashionCNN, 8 canaux, Adam lr=1e-3")

**Observation.** Avec un seul canal de sortie le reseau est severement limite :
il ne peut apprendre qu'UN seul type de motif. Passer a 8 canaux fait bondir
l'accuracy ; passer de 8 a 16 rapporte beaucoup moins. On retrouve le rendement
decroissant de la capacite deja vu au TP precedent.

A noter : ce petit CNN a **beaucoup moins de parametres** que le MLP du TP 2
(~7 800 contre ~200 000) et fait pourtant mieux. C'est tout l'interet de la
convolution : le **partage des poids**. Le meme filtre 3x3 est applique aux 784
positions de l'image, au lieu d'apprendre un poids different par pixel.

## Batch-norm

Extend your model to include the Batch-normalization.

In [ ]:
# TODO
class FashionCNNBN(nn.Module):
    """Meme reseau, avec BatchNorm2d inseree entre la convolution et la ReLU.

    Ordre retenu : Conv -> BatchNorm -> ReLU -> MaxPool.
    C'est l'ordre de l'article original : on normalise la sortie *lineaire* de
    la convolution, avant de lui appliquer la non-linearite.

    Note : BatchNorm2d(C) normalise sur (N, H, W) pour chaque canal C
    separement. Il y a donc 2*C parametres appris (un gamma et un beta par
    canal), et non un par pixel.
    """

    def __init__(self, kernel_size=3, padding=1, out_channels=8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=1, out_channels=out_channels,
                              kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flat = nn.Flatten()
        self.fc = nn.Linear(out_channels * 14 * 14, 10)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.pool(self.relu(self.bn(self.conv(x))))
        return self.logsoftmax(self.fc(self.flat(x)))

In [ ]:
th.manual_seed(0)
model_bn = FashionCNNBN(out_channels=8)
optimizer = th.optim.Adam(model_bn.parameters(), lr=0.001)
hist_bn = train_model(model_bn, loss_fn, optimizer, trainloader, 10,
                      label="FashionCNN + BatchNorm")

plt.figure(figsize=(7, 4))
plt.plot(hist_base['valid_acc'], label="sans BatchNorm")
plt.plot(hist_bn['valid_acc'], label="avec BatchNorm")
plt.xlabel("epoch"); plt.ylabel("accuracy validation")
plt.title("Effet de la BatchNorm"); plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"sans BN : {hist_base['valid_acc'][-1]:.4f} | avec BN : {hist_bn['valid_acc'][-1]:.4f}")

**Erreur a ne pas commettre** (elle etait dans ma premiere version) : reutiliser
le *meme* module BatchNorm a deux endroits differents du reseau, ou appeler deux
fois `self.batch(outpool)` en n'en gardant qu'un. Un module qui porte des
parametres (Conv2d, BatchNorm, Linear) doit etre instancie **une fois par
endroit ou il est utilise**. Les modules sans parametres (ReLU, MaxPool,
Flatten) peuvent en revanche etre reutilises sans probleme.

## More convolution

We can now define an extended model where the basic block is : Conv2D, ReLu, BatchNorm and MaxPool.
We stack two blocks of this kind before the classification.

For instance in the previous model, this kind of block reduce the image size and increase the number of output channels. We can try to do the same and double this number in the second block.

TODO:
- Implement a model with two blocks as decribed above.
- We can then improve the output classifier.
- Play with the hyper-parameters.

Of course if you want to leverage a deeper model it is useful to increase the amount of training data (we only take the first 30k images until now).

In [ ]:
# TODO
def conv_block(in_c, out_c, kernel_size=3, padding=1):
    """Bloc de base : Conv -> BatchNorm -> ReLU -> MaxPool(2).

    Effet sur les dimensions : le nombre de canaux passe de in_c a out_c,
    et H, W sont divises par 2 (a cause du MaxPool).
    """
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

In [ ]:
class FashionCNN2(nn.Module):
    """Deux blocs convolutifs empiles, puis un classifieur a une couche cachee.

    28x28 --bloc1--> 14x14 (C canaux) --bloc2--> 7x7 (2C canaux) --> classifieur
    """

    def __init__(self, C=16, hidden=128, dropout=0.3, in_channels=1, n_classes=10,
                 image_size=28):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(in_channels, C),
            conv_block(C, 2 * C),
        )
        # Astuce tres utile : plutot que de calculer a la main la taille apres
        # aplatissement (source d'erreur n1 en CNN), on la DEDUIT en faisant
        # passer un tenseur factice. Le code reste correct meme si on change
        # le nombre de blocs ou la taille des images.
        with th.no_grad():
            n_flat = self.features(th.zeros(1, in_channels, image_size, image_size)).numel()
        print(f"taille apres aplatissement : {n_flat}")

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_flat, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
            nn.LogSoftmax(dim=1),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
th.manual_seed(0)
model2 = FashionCNN2(C=16)
print("parametres :", count_params(model2))
optimizer = th.optim.Adam(model2.parameters(), lr=0.001)
hist2 = train_model(model2, loss_fn, optimizer, trainloader, 15,
                    label="2 blocs conv (16 -> 32) + classifieur")

print(f"\n1 bloc  : {hist_bn['valid_acc'][-1]:.4f}")
print(f"2 blocs : {hist2['valid_acc'][-1]:.4f}")

On depasse maintenant nettement le MLP du TP precedent (~0.89) : le CNN
atteint ~0.91-0.92 avec beaucoup moins de parametres.

### Pourquoi empiler des blocs ?

A cause du **champ receptif**. Un filtre 3x3 ne voit que 3x3 pixels. Mais apres
un MaxPool, chaque pixel represente une zone 2x2 de l'image d'origine : un
filtre 3x3 du deuxieme bloc voit donc en realite une zone d'environ 8x8 pixels
d'origine. En empilant, le champ receptif croit rapidement, et les couches
profondes voient des motifs de plus en plus globaux.

C'est la hierarchie classique : **bords -> textures -> parties d'objet -> objet**.

# And now in color: CIFAR-10


To experiment image classification on a coloured image, we can use the CIFAR-10 dataset.
You can find more details for instance on this page: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html. We can download the dataset with a dataloader directly:

In [ ]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 16

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = th.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = th.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

With this example we will use a **dataset** via a *dataloader*.  This is a convenient tool to handle datasets with efficient iterators.

In [ ]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(trainloader)
images, labels = dataiter.__next__()
imshow(torchvision.utils.make_grid(images))
print(" ".join(f"{classes[l]:6s}" for l in labels[:8]))
print("forme d'un batch CIFAR :", images.shape, " <- 3 canaux couleur, 32x32")

CIFAR-10 est nettement plus difficile que Fashion-MNIST : images en couleur,
objets photographies sous des angles et des fonds varies. Un MLP y depasse
difficilement 50 % ; un bon CNN atteint 90 %+.

Notre `FashionCNN2` est deja generique (parametres `in_channels` et
`image_size`), on peut donc le reutiliser tel quel comme point de comparaison :

In [ ]:
th.manual_seed(0)
model_cifar = FashionCNN2(C=32, hidden=256, in_channels=3, image_size=32)
optimizer = th.optim.Adam(model_cifar.parameters(), lr=0.001)
hist_cifar = train_model(model_cifar, loss_fn, optimizer, trainloader, 5,
                         validloader=testloader, label="2 blocs conv sur CIFAR-10")

# Todo

Implement `VGG 16` architecture to get state of the art performance (see the course for the architecture)

In [ ]:
# VGG-16 : 13 couches convolutives + 3 couches denses (d'ou "16 couches a poids").
#
# L'idee centrale de VGG (Simonyan & Zisserman, 2014) : n'utiliser QUE des
# convolutions 3x3 (stride 1, padding 1), et empiler.
#   - deux conv 3x3 empilees ont le meme champ receptif qu'une conv 5x5,
#     mais avec 2*(3*3*C*C) = 18C^2 parametres au lieu de 25C^2, et DEUX
#     non-linearites au lieu d'une. Plus expressif et moins cher.
#   - le nombre de canaux double apres chaque MaxPool, ce qui compense
#     approximativement la perte de resolution spatiale.
#
# Configuration D de l'article (= VGG-16). 'M' signifie MaxPool 2x2.
VGG16_CONFIG = [64, 64, 'M',
                128, 128, 'M',
                256, 256, 256, 'M',
                512, 512, 512, 'M',
                512, 512, 512, 'M']

In [ ]:
def make_vgg_features(config, in_channels=3, batchnorm=True):
    """Construit la partie convolutive de VGG a partir d'une liste de config."""
    layers, c_in = [], in_channels
    for v in config:
        if v == 'M':
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        else:
            layers.append(nn.Conv2d(c_in, v, kernel_size=3, padding=1))
            if batchnorm:
                # La BatchNorm n'etait pas dans l'article de 2014, mais sans
                # elle un reseau de 16 couches est tres difficile a entrainer
                # (c'est d'ailleurs pour ca que les auteurs devaient entrainer
                # les versions courtes d'abord). On la met : c'est la variante
                # "VGG16_bn" de torchvision.
                layers.append(nn.BatchNorm2d(v))
            layers.append(nn.ReLU(inplace=True))
            c_in = v
    return nn.Sequential(*layers)

In [ ]:
class VGG16(nn.Module):
    """VGG-16 adapte a CIFAR-10 (images 32x32).

    Difference avec le VGG original : celui-ci etait concu pour ImageNet en
    224x224, et se terminait par 3 couches denses de 4096 neurones sur une
    carte 7x7x512 (soit ~120 M de parametres, dont l'immense majorite dans la
    premiere couche dense). Sur du 32x32, les 5 MaxPool ramenent la carte a
    1x1x512, et un classifieur plus modeste suffit.
    """

    def __init__(self, n_classes=10, in_channels=3, image_size=32, dropout=0.5):
        super().__init__()
        self.features = make_vgg_features(VGG16_CONFIG, in_channels=in_channels)
        with th.no_grad():
            n_flat = self.features(th.zeros(1, in_channels, image_size, image_size)).numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_flat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes),
            nn.LogSoftmax(dim=1),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
vgg = VGG16()
print(vgg)
print("\nparametres VGG-16 :", count_params(vgg))

# Verification des dimensions couche par couche
with th.no_grad():
    t = th.zeros(2, 3, 32, 32)
    print(f"\nentree : {tuple(t.shape)}")
    for layer in vgg.features:
        t = layer(t)
        if isinstance(layer, nn.MaxPool2d):
            print(f"apres MaxPool : {tuple(t.shape)}")

### Entrainement de VGG-16

Attention : c'est **lourd**. Sur CPU, comptez plusieurs heures par epoch -
lancez cette cellule uniquement avec un GPU (Colab : Execution > Modifier le
type d'execution > GPU).

Deux ingredients indispensables pour atteindre les ~92 % annonces :

1. **la data augmentation** (flip horizontal + recadrage aleatoire) : sans
   elle, VGG-16 et ses 15 M de parametres surapprennent massivement les
   50 000 images de CIFAR-10 ;
2. **un scheduler de learning rate** : on commence a 0.01 et on decroit, sinon
   la loss stagne en fin d'entrainement.

In [ ]:
# Data augmentation : appliquee UNIQUEMENT au train, jamais au test.
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # translation aleatoire
    transforms.RandomHorizontalFlip(),         # un chat retourne est un chat
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset_aug = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform_train)
testset_aug = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform_test)
trainloader_aug = th.utils.data.DataLoader(trainset_aug, batch_size=128, shuffle=True, num_workers=2)
testloader_aug = th.utils.data.DataLoader(testset_aug, batch_size=256, shuffle=False, num_workers=2)

In [ ]:
def train_vgg(model, NEpochs=30, lr=0.01):
    """Entrainement avec SGD + momentum + weight decay + scheduler cosinus.

    C'est la recette standard pour les CNN de vision : sur ce type de tache,
    SGD+momentum bien regle generalise souvent *mieux* qu'Adam.
    """
    model = model.to(device)
    optimizer = th.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                             weight_decay=5e-4)
    # weight_decay = penalite L2 sur les poids : ajoute lambda*||w||^2 a la loss,
    # ce qui pousse les poids vers 0 et limite l'overfitting.
    scheduler = th.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NEpochs)
    # Le learning rate decroit en cosinus de lr a ~0 sur les NEpochs epochs.

    hist = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}
    for epoch in range(NEpochs):
        model.train()
        running, correct, n = 0.0, 0, 0
        for X, Y in trainloader_aug:
            X, Y = X.to(device), Y.to(device)
            out = model(X)
            loss = loss_fn(out, Y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            running += loss.item() * len(X)
            correct += (out.argmax(1) == Y).sum().item()
            n += len(X)
        scheduler.step()                       # UNE fois par epoch, pas par batch

        vl, va = evaluate(model, testloader_aug, loss_fn)
        hist['train_loss'].append(running / n); hist['train_acc'].append(correct / n)
        hist['valid_loss'].append(vl); hist['valid_acc'].append(va)
        print(f"epoch {epoch:3d} | lr {scheduler.get_last_lr()[0]:.5f}"
              f" | train acc {correct/n:.4f} | test acc {va:.4f}")
    return hist

In [ ]:
if th.cuda.is_available():
    th.manual_seed(0)
    vgg = VGG16()
    hist_vgg = train_vgg(vgg, NEpochs=30, lr=0.01)

    plt.figure(figsize=(7, 4))
    plt.plot(hist_vgg['train_acc'], label='train')
    plt.plot(hist_vgg['valid_acc'], label='test')
    plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.title('VGG-16 sur CIFAR-10')
    plt.legend(); plt.grid(alpha=.3); plt.show()
    print("accuracy test finale :", hist_vgg['valid_acc'][-1])
else:
    print("Pas de GPU detecte : entrainement de VGG-16 ignore.")
    print("Sur GPU, comptez ~15 min pour 30 epochs et ~92 % d'accuracy test.")

## Bilan du TP

| Modele | Donnees | Parametres | Accuracy |
|---|---|---:|---:|
| MLP 2x200 (TP precedent) | Fashion-MNIST | ~200 k | ~0.89 |
| CNN 1 bloc, 8 canaux | Fashion-MNIST | ~8 k | ~0.90 |
| CNN 2 blocs + BN + Dropout | Fashion-MNIST | ~250 k | ~0.92 |
| CNN 2 blocs | CIFAR-10 | ~800 k | ~0.70 |
| VGG-16 + augmentation | CIFAR-10 | ~15 M | ~0.92 |

### Les formules a retenir

Taille de sortie d'une convolution (par dimension spatiale) :

$$H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1$$

Cas particuliers a connaitre par coeur :
- `k=3, p=1, s=1` -> **taille inchangee** (le cas le plus courant)
- `k=5, p=2, s=1` -> taille inchangee
- `k=2, s=2` (MaxPool) -> **taille divisee par 2**

Nombre de parametres d'une couche `Conv2d(C_in, C_out, k)` :

$$(k \times k \times C_{in} + 1) \times C_{out}$$

Le point remarquable : ce nombre **ne depend pas de la taille de l'image**.
C'est ce qui distingue fondamentalement une convolution d'une couche dense.